In [ ]:
!pip install -U transformers

In [5]:
# ── Questions + Ground Truths ──────────────────────────────────────────────────
QA_PAIRS = [
    {
        "question": "According to the policy, what is considered a wrong use of the company's business trip expense?",
        "ground_truth": "A wrong use of the company's business trip expense occurs when an employee claims reimbursement for a personal trip—such as a salesman returning to Vietnam for a family affair—and attempts to have the company cover all costs, thereby using the business trip expense asset for a non‑business purpose.",
    },
    {
        "question": "What responsibility do staff have regarding the confidentiality of information at FPT Software?",
        "ground_truth": "Staff are responsible for ensuring the confidentiality of all company information, including sales data, project details, technical product information, and personal information of employees.",
    },
    {
        "question": "What approach does FSOFT take to ensure its products and services do not violate human rights with respect to stakeholder input?",
        "ground_truth": "FSOFT incorporates the viewpoints and opinions of third parties in addition to internal stakeholders, and it builds partnerships and collaborates closely with external experts and other relevant stakeholders to prevent human rights violations."
    },
    {
        "question": "When can data processing that goes beyond the purposes of fulfilling a contract be permissible under national data protection legislation?",
        "ground_truth": "Data processing that exceeds the purposes of fulfilling a contract may be permissible if it is authorized through a collective agreement that specifically covers the intended purpose and is drawn up within the parameters of the relevant employment law and national data protection legislation."
    },
    {
        "question": "What does pseudonymisation mean in the context of personal data processing?",
        "ground_truth": "Pseudonymisation is the processing of personal data in such a way that the data cannot be linked to a specific data subject without additional information, which must be stored separately and protected by appropriate technical and organizational measures."
    },
    {
        "question": "Under what circumstances can a data subject exercise the right to be forgotten, and what does this right entail?",
        "ground_truth": "A data subject can exercise the right to be forgotten in certain circumstances by asking the organization to erase the personal data it holds about the individual from its records."
    },
    {
        "question": "What does the abbreviation DPIA stand for in the context of data protection?",
        "ground_truth": "DPIA stands for Data Protection Impact Assessment."
    },
    {
        "question": "What is the business registration number of FPT Corporation?",
        "ground_truth": "FPT Corporation's business registration number is 0101248141."
    },
    {
        "question": "What are the three main sectors in which FPT organizes its business activities?",
        "ground_truth": "FPT organizes its business activities into three main sectors: the Technology sector, the Telecommunications sector, and the Education, Investment and Others sector."
    },
]

In [6]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from concurrent.futures import ThreadPoolExecutor
import time

# ── Device assignment ──────────────────────────────────────────────────────────
device0 = torch.device("cuda:0")
device1 = torch.device("cuda:1")

# ── Load models ────────────────────────────────────────────────────────────────
def load_model(model_id, device):
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
    ).to(device)
    model.eval()
    return processor, model

print("Loading models...")
processor1, model1 = load_model("Qwen/Qwen3.5-4B",             device0)
processor2, model2 = load_model("tnnanh1005/Qwen3.5-4B-SML",   device1)
print("Models loaded.\n")

# ── Inference helper ───────────────────────────────────────────────────────────
def run_inference(processor, model, question: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "user", "content": [{"type": "text", "text": question}]}
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    return processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()

# ── Parallel comparison ────────────────────────────────────────────────────────
def compare_models(qa_pairs: list[dict], max_new_tokens: int = 512) -> list[dict]:
    results = []

    for i, pair in enumerate(qa_pairs, 1):
        question    = pair["question"]
        ground_truth = pair["ground_truth"]

        print(f"{'='*70}")
        print(f"Question {i}/{len(qa_pairs)}: {question}")
        print(f"Ground truth: {ground_truth}")
        print("="*70)

        with ThreadPoolExecutor(max_workers=2) as executor:
            t0 = time.perf_counter()
            future1 = executor.submit(run_inference, processor1, model1, question, max_new_tokens)
            future2 = executor.submit(run_inference, processor2, model2, question, max_new_tokens)
            answer1 = future1.result()
            answer2 = future2.result()
            elapsed = time.perf_counter() - t0

        print(f"\n[Model 1 - Qwen/Qwen3.5-4B]\n{answer1}")
        print(f"\n[Model 2 - tnnanh1005/Qwen3.5-4B-SML]\n{answer2}")
        print(f"\n⏱  Both answers generated in {elapsed:.2f}s\n")

        results.append({
            "id":           i,
            "question":     question,
            "ground_truth": ground_truth,
            "model1":       {"name": "Qwen/Qwen3.5-4B",               "answer": answer1},
            "model2":       {"name": "tnnanh1005/Qwen3.5-4B-SML",     "answer": answer2},
            "elapsed_sec":  round(elapsed, 2),
        })

    return results

results = compare_models(QA_PAIRS)


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Models loaded.

Question 1/9: According to the policy, what is considered a wrong use of the company's business trip expense?
Ground truth: A wrong use of the company's business trip expense occurs when an employee claims reimbursement for a personal trip—such as a salesman returning to Vietnam for a family affair—and attempts to have the company cover all costs, thereby using the business trip expense asset for a non‑business purpose.


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



[Model 1 - Qwen/Qwen3.5-4B]
I don't have access to your company's specific policy documents, as policies vary by organization. To answer your question accurately, please:

1. Check your employee handbook or intranet portal for the "Business Travel Expenses" section
2. Review the finance department's travel guidelines
3. Consult with your HR or finance representative

Common examples of wrong use of business trip expenses typically include:
- Personal expenses disguised as business costs (e.g., dining with non-business contacts)
- Excessive spending beyond approved limits
- Reimbursing personal items (clothing, gifts, etc.)
- Using company funds for non-business activities
- Failing to provide required receipts or documentation
- Traveling without proper approval
- Exceeding approved per diem rates without justification

Would you like me to help you draft a policy question for your HR/finance team, or do you have access to your company's specific policy document you'd like me to revie

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



[Model 1 - Qwen/Qwen3.5-4B]
In the context of data protection and privacy law (specifically under the EU's General Data Protection Regulation, or GDPR), **DPIA** stands for **Data Protection Impact Assessment**.

It is a systematic process used by organizations to identify, analyze, and mitigate potential risks to the rights and freedoms of individuals when processing personal data is likely to result in a high risk. This assessment helps ensure that data protection measures are implemented effectively before the processing activity begins.

[Model 2 - tnnanh1005/Qwen3.5-4B-SML]
DPIA stands for Data Protection Impact Assessment.

⏱  Both answers generated in 7.79s

Question 8/9: What is the business registration number of FPT Corporation?
Ground truth: FPT Corporation's business registration number is 0101248141.


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



[Model 1 - Qwen/Qwen3.5-4B]
FPT Corporation is a large Vietnamese multinational technology company, and its official business registration number (often referred to as the Tax Code or Registration Number) is **0102970000**.

This number is issued by the Vietnam State Administration of Taxation and is used for official tax and legal identification purposes within Vietnam. The company is headquartered in Hanoi and operates globally with subsidiaries in many countries.

[Model 2 - tnnanh1005/Qwen3.5-4B-SML]
The business registration number of FPT Corporation is 0101201201.

⏱  Both answers generated in 7.67s

Question 9/9: What are the three main sectors in which FPT organizes its business activities?
Ground truth: FPT organizes its business activities into three main sectors: the Technology sector, the Telecommunications sector, and the Education, Investment and Others sector.

[Model 1 - Qwen/Qwen3.5-4B]
FPT (FPT Corporation) organizes its business activities primarily into three main 

NameError: name 'json' is not defined

In [7]:
import json
output_path = "/kaggle/working/comparison_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ Results saved to {output_path}")

✅ Results saved to /kaggle/working/comparison_results.json


In [16]:
from rich.table import Table
from rich.console import Console
from rich.text import Text
from PIL import Image
import io

def save_table_as_jpeg(results: list[dict], svg_path: str = "comparison_results.svg", preview_n: int = 3):
    # 1. Render rich table to string buffer
    console = Console(record=True, width=220)  # wider = more readable
    
    sample = results[:preview_n]  # ← only first N samples

    table = Table(
        title="Model Comparison Results",
        show_lines=True,
        header_style="bold cyan",
        width=220,
    )
    table.add_column("#",            style="dim",        width=4,  no_wrap=True)
    table.add_column("Question",     style="bold white", ratio=2)
    table.add_column("Ground Truth", style="green",      ratio=3)
    table.add_column("Model 1",      style="yellow",     ratio=3)
    table.add_column("Model 2",      style="magenta",    ratio=3)

    for r in sample:
        table.add_row(
            str(r["id"]),
            r["question"],
            r["ground_truth"],
            r["model1"]["ground_truth"],
            r["model2"]["ground_truth"],
        )

    console.print(table)

    # 2. Export as SVG → convert to image via Pillow
    svg_content = console.export_svg(title="Model Comparison")

    with open(svg_path, "w", encoding="utf-8") as f:
        f.write(svg_content)
    print(f"✅ SVG saved to {svg_path} — open in browser and screenshot manually.")




# print_comparison_table(results)     # terminal display
save_table_as_jpeg(results)         # → comparison_results.jpg

with open("/kaggle/working/comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

                                                                                                  Model Comparison Results                                                                                                  
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #    ┃ Question                             ┃ Ground Truth                                            ┃ Model 1                                                 ┃ Model 2                                                ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1    │ According to the policy, what is     │ A wrong use of the company's business trip expense      │ I don't have access to your company's specific policy   │ A wrong use of the company's business trip expense     │
│      │ considered a wrong use of the        │ occurs when an employee claims reimbursement for a      │ documents, as policies vary by organization. To answer  │ occurs when an employee claims reimbursement for a     │
│      │ company's business trip expense?     │ personal trip—such as a salesman returning to Vietnam   │ your question accurately, please:                       │ personal trip and tries to have the company cover the  │
│      │                                      │ for a family affair—and attempts to have the company    │                                                         │ cost.                                                  │
│      │                                      │ cover all costs, thereby using the business trip        │ 1. Check your employee handbook or intranet portal for  │                                                        │
│      │                                      │ expense asset for a non‑business purpose.               │ the "Business Travel Expenses" section                  │                                                        │
│      │                                      │                                                         │ 2. Review the finance department's travel guidelines    │                                                        │
│      │                                      │                                                         │ 3. Consult with your HR or finance representative       │                                                        │
│      │                                      │                                                         │                                                         │                                                        │
│      │                                      │                                                         │ Common examples of wrong use of business trip expenses  │                                                        │
│      │                                      │                                                         │ typically include:                                      │                                                        │
│      │                                      │                                                         │ - Personal expenses disguised as business costs (e.g.,  │                                                        │
│      │                                      │                                                         │ dining with non-business contacts)                      │                                                        │
│      │                                      │                                                         │ - Excessive spending beyond approved limits             │                                                        │
│      │              

✅ SVG saved to comparison_results.svg — open in browser and screenshot manually.
